# Bitcoin Wallet Risk Analyzer
**GNN-based Bitcoin wallet classification (Benign / Criminal)**

This notebook is fully self-contained. Run all cells top-to-bottom.  
First run will install dependencies (~2 min).

# Part 1 — Setup
Install dependencies, define the model, and prepare helper functions.  
**Run all cells in this section once** (first run installs packages, ~2 min).

## 1. Install Dependencies

In [ ]:
import subprocess, sys

def install(packages):
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkg.split())

# Core
install(["numpy", "requests", "matplotlib"])

# PyTorch (CPU) — skip if already installed
try:
    import torch
    print(f"PyTorch {torch.__version__} already installed.")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "torchvision",
                           "--index-url", "https://download.pytorch.org/whl/cpu"])
    import torch
    print(f"Installed PyTorch {torch.__version__}")

# PyTorch Geometric
try:
    import torch_geometric
    print(f"PyG {torch_geometric.__version__} already installed.")
except ImportError:
    install(["torch-geometric"])
    import torch_geometric
    print(f"Installed PyG {torch_geometric.__version__}")

# networkx for graph visualization
install(["networkx"])

print("All dependencies ready.")

## 2. Imports

In [ ]:
import math
import os
import sys
import time
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Tuple

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, LayerNorm

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import networkx as nx

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

print(f"Python  {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")
print(f"Device  {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 3. Constants & Model Definition

In [ ]:
# ── Feature columns (must match trained model) ──────────────────────────────
FEATURE_COLUMNS = [
    "lifetime_seconds_log",
    "activity_rate_log",
    "in_out_balance_log",
    "total_txs_log",
    "send_receive_ratio_log",
    "fee_per_tx_log",
    "blocks_btwn_txs_mean_log",
    "fee_share_mean_log",
    "avg_tx_size_log",
    "tx_size_range_log",
    "max_sent_log",
    "max_received_log",
]

FEATURE_DISPLAY_NAMES = [
    "lifetime seconds",
    "activity rate",
    "in/out balance",
    "total txs",
    "send/receive ratio",
    "fee per tx",
    "blocks btwn txs mean",
    "fee share mean",
    "avg tx size",
    "tx size range",
    "max sent",
    "max received",
]

NUM_NODE_FEATURES = 12
NUM_EDGE_FEATURES = 3  # amount_log, direction, timestamp_norm

TIMESTAMP_MIN = 1231006505   # Bitcoin genesis: 2009-01-03
TIMESTAMP_MAX = 1893456000   # ~2030-01-01

MEMPOOL_API = "https://mempool.space/api"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# ── Color palette ───────────────────────────────────────────────────────────
CLR_PURPLE  = "#8b5cf6"
CLR_GREEN   = "#22c55e"
CLR_RED     = "#ef4444"
CLR_ORANGE  = "#f59e0b"
CLR_BLUE    = "#3b82f6"
CLR_GRAY200 = "#e5e7eb"
CLR_GRAY500 = "#6b7280"
CLR_GRAY700 = "#374151"

In [ ]:
# ── OptimalBitcoinGNN ────────────────────────────────────────────────────────
# Must match the class used to train outputs/gnn_model.pt — keep in sync with
# standalone/wallet_analyzer.py and src/models/optimal_gnn.py.

from torch_geometric.nn import global_mean_pool, global_max_pool
from torch_geometric.utils import dropout_edge


class OptimalBitcoinGNN(nn.Module):
    """3-layer GATv2Conv with ghost-node handling, hybrid readout, and DropEdge."""

    def __init__(
        self,
        num_node_features: int = 12,
        num_edge_features: int = 3,
        hidden_dim: int = 64,
        num_heads_1: int = 4,
        num_heads_2: int = 4,
        num_heads_3: int = 2,
        dropout: float = 0.2,
        final_dropout: float = 0.3,
        drop_edge_rate: float = 0.1,
    ):
        super().__init__()
        self.num_node_features = num_node_features
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.drop_edge_rate = drop_edge_rate

        self.ghost_embedding = nn.Parameter(torch.randn(1, num_node_features) * 0.01)
        self.node_type_embedding = nn.Embedding(2, 8)
        self.input_proj = nn.Linear(num_node_features + 1 + 8, num_node_features)

        self.conv1 = GATv2Conv(num_node_features, hidden_dim, heads=num_heads_1,
                               edge_dim=num_edge_features, concat=True, dropout=dropout)
        self.norm1 = LayerNorm(hidden_dim * num_heads_1)

        self.conv2 = GATv2Conv(hidden_dim * num_heads_1, hidden_dim, heads=num_heads_2,
                               edge_dim=num_edge_features, concat=True, dropout=dropout)
        self.norm2 = LayerNorm(hidden_dim * num_heads_2)

        self.residual_proj = nn.Linear(hidden_dim * num_heads_1, hidden_dim * num_heads_2)

        self.conv3 = GATv2Conv(hidden_dim * num_heads_2, hidden_dim // 2, heads=num_heads_3,
                               edge_dim=num_edge_features, concat=True, dropout=dropout * 0.5)
        self.norm3 = LayerNorm((hidden_dim // 2) * num_heads_3)

        final_gnn_dim = (hidden_dim // 2) * num_heads_3
        self.initial_proj = nn.Linear(num_node_features, final_gnn_dim)

        classifier_input_dim = final_gnn_dim * 3  # center + mean + max
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ELU(),
            nn.Dropout(final_dropout),
            nn.Linear(hidden_dim, 2),
        )
        self.dropout_layer = nn.Dropout(dropout)
        self.dropout_light = nn.Dropout(dropout * 0.5)

    def _prepare_node_features(self, x):
        has_features = (x.abs().sum(dim=1) > 0).float()
        ghost_mask = has_features == 0
        if ghost_mask.any():
            x = x.clone()
            x[ghost_mask] = self.ghost_embedding.expand(ghost_mask.sum(), -1)
        node_types = ghost_mask.long()
        type_emb = self.node_type_embedding(node_types)
        x_aug = torch.cat([x, has_features.unsqueeze(1), type_emb], dim=1)
        return self.input_proj(x_aug)

    def forward(self, x, edge_index, edge_attr, batch):
        x = self._prepare_node_features(x)
        x_init = x

        if self.training and self.drop_edge_rate > 0 and edge_index.size(1) > 0:
            edge_index_drop, edge_mask = dropout_edge(edge_index, p=self.drop_edge_rate, training=self.training)
            edge_attr_drop = edge_attr[edge_mask] if edge_attr is not None else None
        else:
            edge_index_drop, edge_attr_drop = edge_index, edge_attr

        h = F.elu(self.norm1(self.conv1(x, edge_index_drop, edge_attr=edge_attr_drop)))
        h = self.dropout_layer(h)
        h_res = h
        h = F.elu(self.norm2(self.conv2(h, edge_index_drop, edge_attr=edge_attr_drop)))
        h = self.dropout_layer(h)
        h = h + self.residual_proj(h_res)
        h = F.elu(self.norm3(self.conv3(h, edge_index, edge_attr=edge_attr)))
        h = self.dropout_light(h)
        h = h + self.initial_proj(x_init)

        center_emb = self._get_center_embeddings(h, batch)
        mean_emb = global_mean_pool(h, batch)
        max_emb = global_max_pool(h, batch)
        h_readout = torch.cat([center_emb, mean_emb, max_emb], dim=1)
        return self.classifier(h_readout)

    def _get_center_embeddings(self, h, batch):
        counts = torch.bincount(batch)
        ptr = torch.zeros(counts.size(0) + 1, dtype=torch.long, device=batch.device)
        torch.cumsum(counts, dim=0, out=ptr[1:])
        return h[ptr[:-1]]


print("Model class defined.")

## 4. Ego-Graph Builder

In [ ]:
class EgoGraphBuilder:
    """Builds 1-hop ego-graphs (matches the training pipeline used for outputs/gnn_model.pt)."""

    SAT_TO_BTC = 1e-8
    SECONDS_PER_DAY = 86400

    def __init__(self):
        self.num_features = NUM_NODE_FEATURES

    def _norm_ts(self, ts):
        if ts <= TIMESTAMP_MIN: return 0.0
        if ts >= TIMESTAMP_MAX: return 1.0
        return (ts - TIMESTAMP_MIN) / (TIMESTAMP_MAX - TIMESTAMP_MIN)

    def _log_amt(self, sats):
        return np.log1p(max(0, sats))

    def _parse_txs(self, center, txs):
        neighbors, edges = set(), []
        for tx in txs:
            if not tx.get("status", {}).get("confirmed", False): continue
            t = tx["status"].get("block_time", 0)
            if t == 0: continue
            ts = self._norm_ts(t)
            is_sender = any(((inp.get("prevout") or {}).get("scriptpubkey_address") == center)
                            for inp in tx.get("vin", []))
            received = sum(o.get("value", 0) for o in tx.get("vout", [])
                           if o.get("scriptpubkey_address") == center)
            if is_sender:
                for o in tx.get("vout", []):
                    r = o.get("scriptpubkey_address"); a = o.get("value", 0)
                    if r and r != center and a > 0:
                        neighbors.add(r); edges.append((r, "out", self._log_amt(a), ts))
            if received > 0:
                for inp in tx.get("vin", []):
                    s = (inp.get("prevout") or {}).get("scriptpubkey_address")
                    if s and s != center:
                        neighbors.add(s); edges.append((s, "in", self._log_amt(received), ts))
        return list(neighbors), edges

    def compute_features(self, address, txs):
        """Match standalone/wallet_analyzer.py:compute_features_from_transactions exactly:
        amounts in BTC (not satoshis), training-time clips, tx_size_range over sent only."""
        total_sent = total_received = total_fees = 0
        n_send = n_recv = 0
        sent_amts, recv_amts, times = [], [], []
        for tx in txs:
            if not tx.get("status", {}).get("confirmed", False): continue
            t = tx["status"].get("block_time", 0)
            if t > 0: times.append(t)
            is_sender = any(((inp.get("prevout") or {}).get("scriptpubkey_address") == address)
                            for inp in tx.get("vin", []))
            recv_in_tx = sum(o.get("value", 0) for o in tx.get("vout", [])
                             if o.get("scriptpubkey_address") == address)
            if is_sender:
                n_send += 1
                for o in tx.get("vout", []):
                    if o.get("scriptpubkey_address") != address:
                        a = o.get("value", 0); total_sent += a
                        if a > 0: sent_amts.append(a)
                total_fees += tx.get("fee", 0)
            if recv_in_tx > 0:
                n_recv += 1; total_received += recv_in_tx; recv_amts.append(recv_in_tx)

        # Convert to BTC to match training scale
        total_sent *= self.SAT_TO_BTC
        total_received *= self.SAT_TO_BTC
        total_fees *= self.SAT_TO_BTC
        sent_amts = [a * self.SAT_TO_BTC for a in sent_amts]
        recv_amts = [a * self.SAT_TO_BTC for a in recv_amts]

        total_txs = n_send + n_recv
        lifetime = (max(times) - min(times)) if len(times) >= 2 else 0
        activity_rate = np.clip(
            (total_txs / max(lifetime, 1)) * self.SECONDS_PER_DAY, 0, 1000
        )
        send_receive_ratio = np.clip(total_sent / max(total_received, 1e-10), 0, 100)
        in_out_balance = np.clip(n_recv / max(n_send, 1e-10), 0, 100)
        fee_per_tx = total_fees / max(total_txs, 1)
        if len(times) >= 2:
            ts = sorted(times)
            blocks_btwn_txs_mean = np.mean([ts[i+1]-ts[i] for i in range(len(ts)-1)]) / 600
        else:
            blocks_btwn_txs_mean = 0
        total_vol = total_sent + total_received
        fee_share_mean = np.clip(total_fees / max(total_vol, 1e-10), 0, 1)
        avg_tx_size = total_vol / max(total_txs, 1)
        tx_size_range = (max(sent_amts) - min(sent_amts)) if sent_amts else 0
        max_sent = max(sent_amts) if sent_amts else 0
        max_received = max(recv_amts) if recv_amts else 0

        raw = np.array([
            lifetime, activity_rate, in_out_balance, total_txs, send_receive_ratio,
            fee_per_tx, blocks_btwn_txs_mean, fee_share_mean, avg_tx_size,
            tx_size_range, max_sent, max_received,
        ], dtype=np.float32)
        return np.log1p(np.abs(raw)).astype(np.float32)

    def build(self, address, txs, label=-1):
        feats = self.compute_features(address, txs)
        neighbors, edge_data = self._parse_txs(address, txs)
        nodes = [address] + neighbors
        idx_map = {a: i for i, a in enumerate(nodes)}
        x = np.zeros((len(nodes), self.num_features), dtype=np.float32)
        x[0] = feats
        ei_list, ea_list = [], []
        for nb, d, amt, ts in edge_data:
            ni = idx_map.get(nb)
            if ni is None: continue
            if d == "out":
                ei_list.append([0, ni]); ea_list.append([amt, 1.0, ts])
            else:
                ei_list.append([ni, 0]); ea_list.append([amt, 0.0, ts])
        if ei_list:
            ei = torch.tensor(ei_list, dtype=torch.long).t().contiguous()
            ea = torch.tensor(ea_list, dtype=torch.float)
        else:
            ei = torch.empty((2, 0), dtype=torch.long)
            ea = torch.empty((0, 3), dtype=torch.float)
        y = torch.full((len(nodes),), -1, dtype=torch.long); y[0] = label
        data = Data(x=torch.from_numpy(x), edge_index=ei, edge_attr=ea, y=y, num_nodes=len(nodes))
        data.center_address = address
        data.node_addresses = nodes
        data.num_ghost_nodes = len(nodes) - 1
        data.num_edges = ei.size(1) if ei.numel() > 0 else 0
        return data


print("EgoGraphBuilder defined.")

## 5. Pipeline Functions (API, Inference, Feature Importance)

In [ ]:
MODEL_DOWNLOAD_URL = (
    "https://github.com/NehorayChalfon0166/final_project/releases/download/v1.0.0/gnn_model.pt"
)

def download_model(dest):
    """Download the trained GNN model from GitHub Releases."""
    print(f"Model not found locally. Downloading...")
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    r = requests.get(MODEL_DOWNLOAD_URL, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    dl = 0
    with open(dest, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk); dl += len(chunk)
            if total:
                print(f"\rDownloading model... {dl*100//total}%", end="", flush=True)
    print(f"\rModel downloaded ({dl//1024} KB)              ")
    return dest


def get_model_path():
    """Resolve the model path — checks several standard locations."""
    candidates = [
        os.path.join(os.getcwd(), "outputs", "gnn_model.pt"),
        os.path.join(os.getcwd(), "..", "outputs", "gnn_model.pt"),
        os.path.join(os.getcwd(), "gnn_model.pt"),
    ]
    # Also check relative to notebook file if __file__ not defined
    for c in candidates:
        if os.path.isfile(c):
            return c
    # Auto-download
    default = os.path.join(os.getcwd(), "outputs", "gnn_model.pt")
    return download_model(default)


def load_temperature(model_path, device):
    """Load calibration temperature saved next to the model. Returns 1.0 if absent."""
    temp_path = os.path.join(os.path.dirname(model_path), "temperature.pt")
    if not os.path.isfile(temp_path):
        return 1.0
    temp_data = torch.load(temp_path, map_location=device, weights_only=True)
    # Saved either as {"temperature": float} or as a bare tensor
    if isinstance(temp_data, dict) and "temperature" in temp_data:
        t = temp_data["temperature"]
    else:
        t = temp_data
    return float(t.item()) if hasattr(t, "item") else float(t)


def fetch_transactions(address):
    """Fetch confirmed transactions from mempool.space."""
    url = f"{MEMPOOL_API}/address/{address}/txs"
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code == 200:
        return r.json()
    elif r.status_code == 400:
        raise ValueError(f"Invalid Bitcoin address: {address}")
    else:
        raise ConnectionError(f"mempool.space returned HTTP {r.status_code}")


def fetch_balance(address):
    """Fetch address balance & stats."""
    try:
        r = requests.get(f"{MEMPOOL_API}/address/{address}", headers=HEADERS, timeout=10)
        if r.status_code != 200: return {}
        d = r.json()
        chain, mem = d.get("chain_stats", {}), d.get("mempool_stats", {})
        funded = chain.get("funded_txo_sum", 0) + mem.get("funded_txo_sum", 0)
        spent  = chain.get("spent_txo_sum", 0)  + mem.get("spent_txo_sum", 0)
        return {
            "balance_btc": (funded - spent) / 1e8,
            "total_received_btc": funded / 1e8,
            "total_sent_btc": spent / 1e8,
            "tx_count": chain.get("tx_count", 0),
            "funded_txo_count": chain.get("funded_txo_count", 0),
            "spent_txo_count": chain.get("spent_txo_count", 0),
        }
    except Exception:
        return {}


def run_inference(graph_data, model_path):
    """GNN inference on a single ego-graph (with temperature scaling if available)."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = OptimalBitcoinGNN(NUM_NODE_FEATURES, NUM_EDGE_FEATURES)
    sd = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(sd); model.to(device); model.eval()
    temperature = load_temperature(model_path, device)
    with torch.no_grad():
        x  = graph_data.x.to(device)
        ei = graph_data.edge_index.to(device)
        ea = graph_data.edge_attr.to(device) if graph_data.edge_attr.numel() > 0 else None
        batch = torch.zeros(graph_data.num_nodes, dtype=torch.long, device=device)
        out = model(x, ei, ea, batch)
        logits = out[0].cpu() / temperature
        probs = F.softmax(logits, dim=0).numpy()
    risk = float(probs[1])
    return {
        "classification": "criminal" if risk > 0.5 else "benign",
        "risk_score": risk,
        "confidence": abs(risk - 0.5) * 2,
        "prob_benign": float(probs[0]),
        "prob_criminal": float(probs[1]),
        "temperature": temperature,
    }


def compute_feature_importance(graph_data, model_path):
    """Gradient-based feature importance."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = OptimalBitcoinGNN(NUM_NODE_FEATURES, NUM_EDGE_FEATURES)
    sd = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(sd); model.to(device); model.eval()
    x  = graph_data.x.clone().requires_grad_(True).to(device)
    ei = graph_data.edge_index.to(device)
    ea = graph_data.edge_attr.to(device)
    batch = torch.zeros(graph_data.num_nodes, dtype=torch.long, device=device)
    out = model(x, ei, ea, batch)
    out[0, 1].backward()
    grads = x.grad[0].abs().cpu().numpy()
    total = grads.sum()
    if total > 0: grads = grads / total
    return {name: float(grads[i]) * 100 for i, name in enumerate(FEATURE_DISPLAY_NAMES)}

print("Pipeline functions defined.")

# Part 2 — Analysis
Enter a wallet address and run the analysis pipeline.  
**Re-run from here** to analyze a different wallet.

## 6. Enter Wallet Address
Run this cell — it will prompt you for an address.

In [ ]:
WALLET_ADDRESS = input("Enter Bitcoin wallet address: ").strip()
if not WALLET_ADDRESS:
    WALLET_ADDRESS = "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa"
    print(f"No input — using default: {WALLET_ADDRESS}")
else:
    print(f"Target: {WALLET_ADDRESS}")

## 7. Run Analysis

In [ ]:
print("=" * 56)
print("  BITCOIN WALLET RISK ANALYZER")
print("=" * 56)
print(f"  Wallet: {WALLET_ADDRESS}\n")

# Step 1: Fetch transactions
print("[1/5] Fetching transactions...", end=" ", flush=True)
transactions = fetch_transactions(WALLET_ADDRESS)
num_txs = len(transactions)
print(f"Found {num_txs} transactions")

# Step 2: Fetch balance
print("[2/5] Fetching balance...", end=" ", flush=True)
balance = fetch_balance(WALLET_ADDRESS)
print("Done")

# Step 3: Build ego-graph
print("[3/5] Building ego-graph...", end=" ", flush=True)
builder = EgoGraphBuilder()
graph_data = builder.build(WALLET_ADDRESS, transactions)
print(f"{graph_data.num_nodes} nodes, {graph_data.num_edges} edges")

# Step 4: GNN inference
print("[4/5] Running GNN inference...", end=" ", flush=True)
model_path = get_model_path()
inference = run_inference(graph_data, model_path)
print("Done")

# Step 5: Feature importance
print("[5/5] Computing feature importance...", end=" ", flush=True)
try:
    importance = compute_feature_importance(graph_data, model_path)
    print("Done")
except Exception as e:
    print(f"Warning: {e}")
    importance = {name: 0.0 for name in FEATURE_DISPLAY_NAMES}

print("\n" + "=" * 56)

## 8. Results Summary

In [ ]:
risk_pct = inference["risk_score"] * 100
if risk_pct <= 25:   level = "LOW"
elif risk_pct <= 50: level = "MODERATE"
elif risk_pct <= 75: level = "HIGH"
else:                level = "CRITICAL"

print(f"  Wallet         : {WALLET_ADDRESS}")
print(f"  Classification : {inference['classification'].upper()}")
print(f"  Risk Score     : {risk_pct:.1f}%  ({level})")
print(f"  Confidence     : {inference['confidence'] * 100:.1f}%")
print()
if balance:
    print(f"  Balance        : {balance.get('balance_btc', 0):.8f} BTC")
    print(f"  Total Received : {balance.get('total_received_btc', 0):.8f} BTC")
    print(f"  Total Sent     : {balance.get('total_sent_btc', 0):.8f} BTC")
    print(f"  Transactions   : {balance.get('tx_count', num_txs)}")
    print()

print(f"  Graph Stats    : {graph_data.num_nodes} nodes, {graph_data.num_edges} edges, "
      f"{graph_data.num_ghost_nodes} neighbors")
print()

sorted_feats = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)
max_val = sorted_feats[0][1] if sorted_feats else 1
max_name_len = max(len(n) for n, _ in sorted_feats[:6])
print("  Top Features:")
for i, (name, val) in enumerate(sorted_feats[:6], 1):
    bar = "#" * int((val / max(max_val, 0.01)) * 20)
    print(f"  {i}. {name:<{max_name_len}}  {val:>5.1f}%  {bar}")
print("\n" + "=" * 56)

In [ ]:
from IPython.display import HTML, display

cls = inference["classification"].upper()
risk_pct = inference["risk_score"] * 100
benign_pct = inference["prob_benign"] * 100
criminal_pct = inference["prob_criminal"] * 100
conf = inference["confidence"] * 100

if cls == "CRIMINAL":
    main_color, bg, border = "#ef4444", "#fef2f2", "#fca5a5"
    icon = "&#9888;"
else:
    main_color, bg, border = "#22c55e", "#f0fdf4", "#86efac"
    icon = "&#10003;"

display(HTML(f"""
<div style="border:2px solid {border}; border-radius:12px; background:{bg};
            padding:20px 28px; margin:10px 0; font-family:sans-serif; max-width:500px;">
  <div style="font-size:36px; font-weight:bold; color:{main_color}; text-align:center;">
    {icon} {cls}
  </div>
  <div style="text-align:center; margin:6px 0 12px 0; font-size:11px; color:#9ca3af;
              word-break:break-all; font-family:monospace;">{WALLET_ADDRESS}</div>
  <div style="margin:0 0 8px 0; background:linear-gradient(to right, #22c55e {benign_pct:.1f}%, #ef4444 {benign_pct:.1f}%);
              border-radius:8px; height:28px; overflow:hidden;"></div>
  <div style="display:flex; justify-content:space-between; font-size:13px; color:#6b7280;">
    <span>Benign {benign_pct:.1f}%</span>
    <span>Criminal {criminal_pct:.1f}%</span>
  </div>
  <div style="text-align:center; margin-top:10px; font-size:13px; color:#6b7280;">
    Confidence: {conf:.1f}%
  </div>
</div>
"""))

## 9. Transaction Volume Over Time

In [ ]:
# Aggregate into weekly buckets
weekly = {}
for tx in transactions:
    if not tx.get("status", {}).get("confirmed", False): continue
    t = tx["status"].get("block_time", 0)
    if t == 0: continue
    dt = datetime.fromtimestamp(t, tz=timezone.utc)
    wk = (dt - timedelta(days=dt.weekday())).strftime("%Y-%m-%d")
    if wk not in weekly: weekly[wk] = {"in": 0.0, "out": 0.0}
    is_sender = any(((inp.get("prevout") or {}).get("scriptpubkey_address") == WALLET_ADDRESS)
                    for inp in tx.get("vin", []))
    received = sum(o.get("value", 0) for o in tx.get("vout", [])
                   if o.get("scriptpubkey_address") == WALLET_ADDRESS)
    if received > 0: weekly[wk]["in"] += received / 1e8
    if is_sender:
        sent = sum(o.get("value", 0) for o in tx.get("vout", [])
                   if o.get("scriptpubkey_address") != WALLET_ADDRESS)
        weekly[wk]["out"] += sent / 1e8

if weekly:
    weeks = sorted(weekly.keys())
    inc = [weekly[w]["in"] for w in weeks]
    out = [weekly[w]["out"] for w in weeks]

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("white")
    x = np.arange(len(weeks)); w = 0.35
    ax.bar(x - w/2, inc, w, label="Received", color=CLR_GREEN, edgecolor="none")
    ax.bar(x + w/2, out, w, label="Sent", color=CLR_RED, edgecolor="none")
    ax.set_ylabel("BTC", fontsize=12, color=CLR_GRAY500)
    ax.set_title("Transaction Volume (Weekly)", fontsize=16, fontweight="bold",
                 color=CLR_GRAY700, pad=15)
    if len(weeks) > 20:
        step = max(1, len(weeks) // 10)
        ticks = list(range(0, len(weeks), step))
        ax.set_xticks([x[i] for i in ticks])
        ax.set_xticklabels([weeks[i] for i in ticks], rotation=45, ha="right",
                           fontsize=9, color=CLR_GRAY500)
    else:
        ax.set_xticks(x)
        ax.set_xticklabels(weeks, rotation=45, ha="right", fontsize=9, color=CLR_GRAY500)
    ax.tick_params(axis="y", colors=CLR_GRAY500, labelsize=10)
    ax.grid(axis="y", ls="--", alpha=0.3, color=CLR_GRAY200); ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(CLR_GRAY200); ax.spines["bottom"].set_color(CLR_GRAY200)
    ax.legend(fontsize=11, frameon=False)
    ax.text(0.5, -0.22, f"Total Received: {sum(inc):.8f} BTC  |  Total Sent: {sum(out):.8f} BTC",
            transform=ax.transAxes, ha="center", fontsize=10, color=CLR_GRAY500)
    plt.tight_layout(); plt.show()
else:
    print("No confirmed transactions to chart.")

## 10. Transaction Frequency Over Time

In [ ]:
# Aggregate into monthly buckets
monthly_counts = {}
for tx in transactions:
    if not tx.get("status", {}).get("confirmed", False): continue
    t = tx["status"].get("block_time", 0)
    if t == 0: continue
    dt = datetime.fromtimestamp(t, tz=timezone.utc)
    key = dt.strftime("%Y-%m")
    monthly_counts[key] = monthly_counts.get(key, 0) + 1

if monthly_counts:
    months = sorted(monthly_counts.keys())
    counts = [monthly_counts[m] for m in months]

    fig, ax = plt.subplots(figsize=(12, 4))
    fig.patch.set_facecolor("white")
    ax.bar(range(len(months)), counts, color=CLR_ORANGE, edgecolor="none", width=0.8)
    ax.set_ylabel("Transactions", fontsize=12, color=CLR_GRAY500)
    ax.set_title("Transaction Frequency (Monthly)", fontsize=16, fontweight="bold",
                 color=CLR_GRAY700, pad=15)
    if len(months) > 20:
        step = max(1, len(months) // 10)
        ticks = list(range(0, len(months), step))
        ax.set_xticks(ticks)
        ax.set_xticklabels([months[i] for i in ticks], rotation=45, ha="right",
                           fontsize=9, color=CLR_GRAY500)
    else:
        ax.set_xticks(range(len(months)))
        ax.set_xticklabels(months, rotation=45, ha="right", fontsize=9, color=CLR_GRAY500)
    ax.tick_params(axis="y", colors=CLR_GRAY500, labelsize=10)
    ax.grid(axis="y", ls="--", alpha=0.3, color=CLR_GRAY200); ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(CLR_GRAY200); ax.spines["bottom"].set_color(CLR_GRAY200)
    ax.text(0.5, -0.22, f"Total: {sum(counts)} transactions over {len(months)} months",
            transform=ax.transAxes, ha="center", fontsize=10, color=CLR_GRAY500)
    plt.tight_layout(); plt.show()
else:
    print("No confirmed transactions to chart.")

## 11. Cumulative Balance Over Time

In [ ]:
# Build time-series of balance changes
events = []  # (timestamp, delta_btc)
for tx in transactions:
    if not tx.get("status", {}).get("confirmed", False): continue
    t = tx["status"].get("block_time", 0)
    if t == 0: continue
    is_sender = any(((inp.get("prevout") or {}).get("scriptpubkey_address") == WALLET_ADDRESS)
                    for inp in tx.get("vin", []))
    received = sum(o.get("value", 0) for o in tx.get("vout", [])
                   if o.get("scriptpubkey_address") == WALLET_ADDRESS) / 1e8
    sent = 0
    if is_sender:
        for inp in tx.get("vin", []):
            pv = inp.get("prevout") or {}
            if pv.get("scriptpubkey_address") == WALLET_ADDRESS:
                sent += pv.get("value", 0) / 1e8
    delta = received - sent
    events.append((t, delta))

if events:
    events.sort(key=lambda e: e[0])
    times_dt = [datetime.fromtimestamp(e[0], tz=timezone.utc) for e in events]
    cum_balance = np.cumsum([e[1] for e in events])

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("white")
    ax.fill_between(times_dt, cum_balance, alpha=0.3, color=CLR_BLUE)
    ax.plot(times_dt, cum_balance, color=CLR_BLUE, lw=2)
    ax.set_ylabel("Balance (BTC)", fontsize=12, color=CLR_GRAY500)
    ax.set_title("Cumulative Balance Over Time", fontsize=16, fontweight="bold",
                 color=CLR_GRAY700, pad=15)
    ax.tick_params(axis="both", colors=CLR_GRAY500, labelsize=10)
    ax.grid(axis="y", ls="--", alpha=0.3, color=CLR_GRAY200); ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(CLR_GRAY200); ax.spines["bottom"].set_color(CLR_GRAY200)
    fig.autofmt_xdate()
    ax.text(0.5, -0.18, f"Current Balance: {cum_balance[-1]:.8f} BTC",
            transform=ax.transAxes, ha="center", fontsize=11, color=CLR_GRAY500)
    plt.tight_layout(); plt.show()
else:
    print("No events to chart.")

## 12. Ego-Graph — 5 Most Recent Transactions Classified

In [ ]:
# Get the 5 most recent confirmed transactions
print("Building ego-graph from 5 most recent transactions...")
recent_txs = sorted(
    [tx for tx in transactions if tx.get("status", {}).get("confirmed", False)
     and tx["status"].get("block_time", 0) > 0],
    key=lambda tx: tx["status"]["block_time"], reverse=True
)[:5]

# Collect unique neighbor addresses from only these 5 txs
recent_neighbors = set()
recent_edges = []  # (neighbor_addr, direction)
for tx in recent_txs:
    is_sender = any(((inp.get("prevout") or {}).get("scriptpubkey_address") == WALLET_ADDRESS)
                    for inp in tx.get("vin", []))
    if is_sender:
        for o in tx.get("vout", []):
            addr = o.get("scriptpubkey_address")
            if addr and addr != WALLET_ADDRESS:
                recent_neighbors.add(addr)
                recent_edges.append((addr, "out"))
    for inp in tx.get("vin", []):
        addr = (inp.get("prevout") or {}).get("scriptpubkey_address")
        if addr and addr != WALLET_ADDRESS:
            recent_neighbors.add(addr)
            recent_edges.append((addr, "in"))

# Classify each neighbor through the model
print(f"Classifying {len(recent_neighbors)} neighbors...")
neighbor_classifications = {}
model_path_ego = get_model_path()
for i, addr in enumerate(list(recent_neighbors)):
    try:
        nb_txs = fetch_transactions(addr)
        time.sleep(0.5)  # rate limiting
        if len(nb_txs) < 2:
            neighbor_classifications[addr] = "unknown"
            continue
        nb_graph = builder.build(addr, nb_txs)
        nb_result = run_inference(nb_graph, model_path_ego)
        neighbor_classifications[addr] = nb_result["classification"]
        print(f"  [{i+1}/{len(recent_neighbors)}] {addr[:12]}... -> {nb_result['classification'].upper()}")
    except Exception as e:
        neighbor_classifications[addr] = "unknown"
        print(f"  [{i+1}/{len(recent_neighbors)}] {addr[:12]}... -> error: {e}")

# Build graph with ONLY center + recent neighbors
G = nx.DiGraph()
nodes_list = [WALLET_ADDRESS] + list(recent_neighbors)
for i, addr in enumerate(nodes_list):
    G.add_node(i, is_center=(i == 0), address=addr)

idx_map = {a: i for i, a in enumerate(nodes_list)}
seen_edges = set()
for nb_addr, direction in recent_edges:
    ni = idx_map.get(nb_addr)
    if ni is None:
        continue
    if direction == "out":
        edge_key = (0, ni)
    else:
        edge_key = (ni, 0)
    if edge_key not in seen_edges:
        G.add_edge(*edge_key)
        seen_edges.add(edge_key)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor("white")
pos = nx.spring_layout(G, k=2.5 / max(1, G.number_of_nodes() ** 0.5), seed=42, iterations=50)

# Draw edges
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=CLR_GRAY200, alpha=0.6,
                       arrows=True, arrowsize=12, width=1.2,
                       connectionstyle="arc3,rad=0.1")

# Node colors: center = its classification, neighbors = their classification
center_cls = inference["classification"]
node_colors = []
node_sizes = []
for n in G.nodes():
    addr = G.nodes[n].get("address", "")
    if G.nodes[n].get("is_center"):
        node_colors.append(CLR_RED if center_cls == "criminal" else CLR_GREEN)
        node_sizes.append(500)
    else:
        cls = neighbor_classifications.get(addr, "unknown")
        if cls == "criminal":
            node_colors.append(CLR_RED)
        elif cls == "benign":
            node_colors.append(CLR_GREEN)
        else:
            node_colors.append(CLR_GRAY500)
        node_sizes.append(250)

nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes,
                       edgecolors="white", linewidths=1.5)

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
ax.set_title(f"Ego-Graph \u2014 5 Most Recent Transactions\n{n_nodes} nodes, {n_edges} edges",
             fontsize=13, fontweight="bold", color=CLR_GRAY700, pad=15)
ax.axis("off")

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CLR_GREEN, markersize=12, label="Benign"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CLR_RED, markersize=12, label="Criminal"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CLR_GRAY500, markersize=8, label="Unknown"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=10, frameon=True,
          facecolor="white", edgecolor=CLR_GRAY200)
plt.tight_layout(); plt.show()

## 13. Feature Importance

In [ ]:
sorted_imp = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)
names = [item[0] for item in sorted_imp]
values = [item[1] for item in sorted_imp]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("white")
y_pos = np.arange(len(names))
bars = ax.barh(y_pos, values, color=CLR_PURPLE, edgecolor="none", height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=11, color=CLR_GRAY700)
ax.invert_yaxis()
ax.set_xlabel("Importance (%)", fontsize=12, color=CLR_GRAY500)
ax.set_title("Feature Importance (Gradient-based)", fontsize=16, fontweight="bold",
             color=CLR_GRAY700, pad=15)
ax.tick_params(axis="x", colors=CLR_GRAY500, labelsize=10)
ax.grid(axis="x", ls="--", alpha=0.3, color=CLR_GRAY200); ax.set_axisbelow(True)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(CLR_GRAY200); ax.spines["bottom"].set_color(CLR_GRAY200)
for v, bar in zip(values, bars):
    ax.text(v + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{v:.1f}%", va="center", fontsize=9, color=CLR_GRAY500)
plt.tight_layout(); plt.show()

## 14. Transaction History Table

In [ ]:
from IPython.display import HTML, display

def fmt_sats(sats):
    if abs(sats) >= 1_000_000:
        return f"{sats / 1e8:.8f} BTC"
    return f"{sats:,} sats"

def truncate(s, n=12):
    if not s: return ""
    return s[:n] + "..." + s[-6:] if len(s) > n + 6 else s

# Build HTML table
confirmed_txs = [tx for tx in transactions if tx.get("status", {}).get("confirmed", False)]
confirmed_txs.sort(key=lambda tx: tx["status"].get("block_time", 0), reverse=True)

rows = []
for tx in confirmed_txs[:50]:  # show last 50
    txid = tx.get("txid", "")
    t = tx["status"].get("block_time", 0)
    dt_str = datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d %H:%M") if t else ""
    block = tx["status"].get("block_height", "")
    fee = tx.get("fee", 0)
    size = tx.get("size", 0)

    # Determine direction for this wallet
    is_sender = any(((inp.get("prevout") or {}).get("scriptpubkey_address") == WALLET_ADDRESS)
                    for inp in tx.get("vin", []))
    received = sum(o.get("value", 0) for o in tx.get("vout", [])
                   if o.get("scriptpubkey_address") == WALLET_ADDRESS)
    sent_val = 0
    if is_sender:
        sent_val = sum(o.get("value", 0) for o in tx.get("vout", [])
                       if o.get("scriptpubkey_address") != WALLET_ADDRESS)

    if is_sender and received > 0:
        direction = '<span style="color:#f59e0b">SELF</span>'
        amount_str = fmt_sats(received)
    elif is_sender:
        direction = '<span style="color:#ef4444">SENT</span>'
        amount_str = fmt_sats(sent_val)
    else:
        direction = '<span style="color:#22c55e">RECV</span>'
        amount_str = fmt_sats(received)

    link = f'<a href="https://mempool.space/tx/{txid}" target="_blank" style="color:#3b82f6">{truncate(txid)}</a>'
    rows.append(f"<tr><td>{link}</td><td>{direction}</td><td>{amount_str}</td>"
                f"<td>{fmt_sats(fee)}</td><td>{block}</td><td>{dt_str}</td></tr>")

html = f"""
<style>
.tx-table {{ border-collapse: collapse; width: 100%; font-family: monospace; font-size: 12px; }}
.tx-table th {{ background: #f3f4f6; color: #374151; padding: 8px 10px; text-align: left;
               border-bottom: 2px solid #e5e7eb; }}
.tx-table td {{ padding: 6px 10px; border-bottom: 1px solid #f3f4f6; color: #4b5563; }}
.tx-table tr:hover {{ background: #f9fafb; }}
.tx-table a {{ text-decoration: none; }} .tx-table a:hover {{ text-decoration: underline; }}
</style>
<h3 style="color:#374151; font-family:sans-serif;">Transaction History
  <span style="font-weight:normal; font-size:14px; color:#6b7280;">({len(confirmed_txs)} confirmed, showing up to 50)</span>
</h3>
<table class="tx-table">
<tr><th>TXID</th><th>Direction</th><th>Amount</th><th>Fee</th><th>Block</th><th>Date (UTC)</th></tr>
{''.join(rows)}
</table>
"""
display(HTML(html))